In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

print("Libraries loaded")

Libraries loaded


In [3]:
DATA_PATH = '../data/raw/wec_laps.parquet'

df = pd.read_parquet(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Shape: (334980, 45)
Rows: 334,980
Columns: 45

Memory usage: 155.4 MB


In [4]:
print("=== DTYPES ===")
print(df.dtypes)

=== DTYPES ===
series_code                     str
series                          str
start_date                      str
year                          int64
event                           str
session                         str
session_id                    int64
session_time                float64
clock_time                  float64
session_time_lap_number       int64
car                           int64
class                           str
driver_name                     str
driver_id                       str
lap                           int64
lap_time                    float64
lap_time_s1                 float64
lap_time_s2                 float64
lap_time_s3                 float64
lap_time_driver_rank          int64
lap_time_driver_quartile      int64
bpillar_quartile            float64
pit_time                    float64
flags                           str
stint_start                   int64
stint_number                  int64
stint_lap                     int64
license      

In [5]:
df.head(3)

,series_code,series,start_date,year,event,session,session_id,session_time,clock_time,session_time_lap_number,car,class,driver_name,driver_id,lap,lap_time,lap_time_s1,lap_time_s2,lap_time_s3,lap_time_driver_rank,lap_time_driver_quartile,bpillar_quartile,pit_time,flags,stint_start,stint_number,stint_lap,license,license_rank,driver_country,team_name,chassis,homologation,manufacturer,air_temp_f,track_temp_f,humidity_percent,pressure_inhg,wind_speed_mph,wind_direction_degrees,raining,est_tire_age,class_normalized,class_category,same_class
0,wec,wec-2021,2021-06-11 15:15:00,2021,Portimao,practice,684,128.127,55028.127,1,1,LMP2,Tati Calderon,tati calderon,1,128.127,44.636,35.915,47.576,12,4,NaN,8.587,NaN,1,1,0,Silver,3,COL,Richard Mille Racing Team,Oreca 07 - Gibson,LMP2,ORECA,76.280,95.180,77.000,948.400,4.320,41,False,NaN,LMP2,LMP2,True
1,wec,wec-2021,2021-06-11 15:15:00,2021,Portimao,practice,684,604.361,55504.361,5,1,LMP2,Tati Calderon,tati calderon,2,476.234,407.872,32.397,35.965,13,4,NaN,380.462,NaN,0,1,1,Silver,3,COL,Richard Mille Racing Team,Oreca 07 - Gibson,LMP2,ORECA,77.720,97.340,74.000,948.200,3.600,157,False,NaN,LMP2,LMP2,True
2,wec,wec-2021,2021-06-11 15:15:00,2021,Portimao,practice,684,701.560,55601.560,6,1,LMP2,Tati Calderon,tati calderon,3,97.199,30.483,31.523,35.193,6,2,NaN,NaN,NaN,0,1,2,Silver,3,COL,Richard Mille Racing Team,Oreca 07 - Gibson,LMP2,ORECA,78.080,97.520,74.000,948.100,1.800,322,False,NaN,LMP2,LMP2,True


In [6]:
print("=== class_normalized ===")
print(df['class_normalized'].value_counts())

print("\n=== session ===")
print(df['session'].value_counts())

print("\n=== year ===")
print(df['year'].value_counts().sort_index())

print("\n=== homologation ===")
print(df['homologation'].value_counts())

=== class_normalized ===
class_normalized
HYPERCAR    146171
LMP2         98193
GT3          90616
Name: count, dtype: int64

=== session ===
session
race          174955
practice      121455
test           28325
qualifying      9477
warmup           768
Name: count, dtype: int64

=== year ===
year
2021     30644
2022     49911
2023     48601
2024     85205
2025    120619
Name: count, dtype: int64

=== homologation ===
homologation
LMP2    98193
LMH     92846
GT3     90616
LMDh    53325
Name: count, dtype: int64


In [7]:
print("=== lap_time sample values ===")
print(df['lap_time'].dropna().head(10).tolist())

print(f"\nDtype: {df['lap_time'].dtype}")
print(f"Nulls: {df['lap_time'].isna().sum():,}")
print(f"Non-null sample: {df['lap_time'].dropna().iloc[0]!r}")

=== lap_time sample values ===
[128.127, 476.234, 97.199, 96.187, 96.464, 97.06, 96.074, 104.44, 650.079, 96.473]

Dtype: float64
Nulls: 0
Non-null sample: np.float64(128.127)


In [8]:
null_stats = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
}).query('null_count > 0').sort_values('null_pct', ascending=False)

print(f"Columns with nulls: {len(null_stats)}")
print(null_stats)

Columns with nulls: 6
                  null_count  null_pct
bpillar_quartile      334980   100.000
pit_time              301280    89.940
est_tire_age          160025    47.770
flags                  30644     9.150
track_temp_f            1807     0.540
driver_country           443     0.130


In [9]:
print("=== flags value_counts ===")
print(df['flags'].value_counts(dropna=False).head(30))

=== flags value_counts ===
flags
GF     286427
NaN     30644
SF       6854
FCY      4219
FF       4198
RF       2638
Name: count, dtype: int64


In [10]:
print("=== pit_time ===")
print(f"Non-null: {df['pit_time'].notna().sum():,}")
print(f"Zero values: {(df['pit_time'] == 0).sum():,}")
print(f"Positive values: {(df['pit_time'] > 0).sum():,}")
print(f"\nNon-zero sample:")
print(df[df['pit_time'] > 0]['pit_time'].describe())

=== pit_time ===
Non-null: 33,700
Zero values: 0
Positive values: 33,700

Non-zero sample:
count   33700.000
mean      254.719
std       472.914
min         2.087
25%        71.368
50%        95.370
75%       260.894
max     12992.043
Name: pit_time, dtype: float64


In [11]:
for flag in ['SF', 'FF']:
    print(f"\n=== {flag} ===")
    sample = df[df['flags'] == flag][
        ['year', 'event', 'session', 'lap', 'lap_time', 'flags', 'stint_lap']
    ].head(10)
    print(sample.to_string())


=== SF ===
       year  event session  lap  lap_time flags  stint_lap
33015  2022  Monza    race   78   103.194    SF         25
33016  2022  Monza    race   79   113.802    SF         26
33017  2022  Monza    race   80   197.686    SF         27
33018  2022  Monza    race   81   280.657    SF         28
33019  2022  Monza    race   82   253.016    SF         29
33020  2022  Monza    race   83   156.421    SF         30
33021  2022  Monza    race   84   182.846    SF          0
33022  2022  Monza    race   85   113.192    SF          1
33023  2022  Monza    race   86   165.917    SF          2
33162  2022  Monza    race   79   198.211    SF          0

=== FF ===
       year  event   session  lap  lap_time flags  stint_lap
30672  2022  Monza  practice   29   101.787    FF          2
30709  2022  Monza  practice   37   101.838    FF         14
30749  2022  Monza  practice   40   102.173    FF         11
30831  2022  Monza  practice   40   101.342    FF          3
30872  2022  Monza  pr

In [19]:
print("=== est_tire_age null / session ===")
print(df.groupby('session')['est_tire_age'].apply(
    lambda x: f"{x.isna().sum():,} / {len(x):,} ({x.isna().mean()*100:.1f}%)"
))

print("\n=== est_tire_age null / year ===")
print(df.groupby('year')['est_tire_age'].apply(
    lambda x: f"{x.isna().sum():,} / {len(x):,} ({x.isna().mean()*100:.1f}%)"
))

=== est_tire_age null / session ===
session
practice      121,455 / 121,455 (100.0%)
qualifying        9,477 / 9,477 (100.0%)
race                  0 / 174,955 (0.0%)
test            28,325 / 28,325 (100.0%)
warmup                768 / 768 (100.0%)
Name: est_tire_age, dtype: str

=== est_tire_age null / year ===
year
2021     14,833 / 30,644 (48.4%)
2022     21,272 / 49,911 (42.6%)
2023     25,254 / 48,601 (52.0%)
2024     45,971 / 85,205 (54.0%)
2025    52,695 / 120,619 (43.7%)
Name: est_tire_age, dtype: str


In [20]:
print("=== null flags / year ===")
print(df[df['flags'].isna()]['year'].value_counts())

print("\n=== null flags / session ===")
print(df[df['flags'].isna()]['session'].value_counts())

=== null flags / year ===
year
2021    30644
Name: count, dtype: int64

=== null flags / session ===
session
race          15811
practice      13923
qualifying      796
warmup          114
Name: count, dtype: int64


In [21]:
hc = df[(df['class_normalized'] == 'HYPERCAR') & (df['session'] == 'race')]

print("=== HYPERCAR manufacturers / year ===")
pivot = hc.groupby(['year', 'manufacturer']).size().unstack(fill_value=0)
print(pivot.to_string())

=== HYPERCAR manufacturers / year ===
manufacturer  Alpine  Aston Martin   BMW  Cadillac  Ferrari  Glickenhaus  Isotta Fraschini  Lamborghini  Peugeot  Porsche  Toyota  Vanwall
year                                                                                                                                      
2021            1091             0     0         0        0          536                 0            0        0        0    2152        0
2022            1326             0     0         0        0         1136                 0            0     1067        0    2536        0
2023               0             0     0      1109     2112          425                 0            0     2033     3082    2167      897
2024            2410             0  2241      1143     3459            0               478         1024     2293     5666    2314        0
2025            3678          3428  3390      4155     5555            0                 0            0     3667     5842    371

In [22]:
gt3 = df[(df['class_normalized'] == 'GT3') & (df['session'] == 'race')]

print("=== GT3 manufacturers / year ===")
pivot = gt3.groupby(['year', 'manufacturer']).size().unstack(fill_value=0)
print(pivot.to_string())

=== GT3 manufacturers / year ===
manufacturer  Aston Martin   BMW  Chevrolet  Ferrari  Ford  Lamborghini  Lexus  McLaren  Mercedes-AMG  Porsche
year                                                                                                          
2024                  2114  2051       1986     2065  1966         1960   1758     2123             0     2183
2025                  3219  2904       3409     4037  2717            0   2712     2988          3031     3697


In [23]:
for cls, subset in [('HYPERCAR', hc), ('GT3', gt3)]:
    print(f"\n=== {cls}: race laps per manufacturer (all years) ===")
    print(subset.groupby('manufacturer').size().sort_values())


=== HYPERCAR: race laps per manufacturer (all years) ===
manufacturer
Isotta Fraschini      478
Vanwall               897
Lamborghini          1024
Glickenhaus          2097
Aston Martin         3428
BMW                  5631
Cadillac             6407
Alpine               8505
Peugeot              9060
Ferrari             11126
Toyota              12885
Porsche             14590
dtype: int64

=== GT3: race laps per manufacturer (all years) ===
manufacturer
Lamborghini     1960
Mercedes-AMG    3031
Lexus           4470
Ford            4683
BMW             4955
McLaren         5111
Aston Martin    5333
Chevrolet       5395
Porsche         5880
Ferrari         6102
dtype: int64


In [24]:
hc = df[(df['class_normalized'] == 'HYPERCAR') & (df['session'] == 'race')]

print("=== manufacturer → homologation (race only) ===")
print(
    hc.groupby(['manufacturer', 'homologation'])
    .size()
    .reset_index(name='laps')
    .sort_values(['homologation', 'laps'], ascending=[True, False])
    .to_string(index=False)
)

=== manufacturer → homologation (race only) ===
    manufacturer homologation  laps
         Porsche         LMDh 14590
        Cadillac         LMDh  6407
             BMW         LMDh  5631
     Lamborghini         LMDh  1024
          Toyota          LMH 12885
         Ferrari          LMH 11126
         Peugeot          LMH  9060
          Alpine          LMH  8505
    Aston Martin          LMH  3428
     Glickenhaus          LMH  2097
         Vanwall          LMH   897
Isotta Fraschini          LMH   478
